# 03 · Truth generator: who gets worse, and when

**Who this is for:** you, after notebook 02.
**You should already know:** the physiology vs measurement split, and what an *offset* is (notebook 02).

**By the end you will be able to:**
- explain the **hidden rule** that decides who starts to deteriorate, and why the model never sees it
- describe a **deterioration trajectory** (respiratory, sepsis, cardiac) and watch one unfold in the vitals
- explain **recoveries**, **sudden events**, and why a patient's monitor stops after escalation
- build the **"escalated within 6 hours?"** label from outcome events, and see why it's only known 6 hours later
- see how a **protocol change** creates *concept drift*: same vitals, different outcomes

**Outline**
1. Setup
2. The four steps
3. The hidden rule
4. Simulate 3 days of the ward
5. Anatomy of one deterioration
6. Three ways to deteriorate
7. Recoveries and sudden events
8. Building the 6-hour label
9. Concept drift preview: a new escalation protocol
10. Exercise: how much warning do we get?
11. Pitfalls and next steps

Code used: `generator/truth.py` (tests in `tests/test_truth.py`). Plain-language background: `ml_model.md` §5.

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from generator import hospital, synthea, truth


def find_repo_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for folder in [start, *start.parents]:
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found")


ROOT = find_repo_root()
RUN = synthea.SyntheaRun()
csv_dir = synthea.run_synthea(RUN, ROOT / "data" / "synthea" / f"p{RUN.population}_s{RUN.seed}")
profiles = synthea.build_profiles(
    synthea.load_tables(csv_dir), pd.Timestamp(RUN.reference_date, tz="UTC")
)
inpatients = hospital.admit_inpatients(profiles)
START = hospital.SIM_START
print(len(inpatients), "inpatients")

## 2. The four steps

```
undisturbed true vitals ──► 1. hidden rule: chance of starting to deteriorate, every patient, every hour
                                   │ (a few patients "roll" a start)
                                   ▼
                            2. trajectory: 2-12 h of worsening physiology (offsets)
                                   │
                    ┌──────────────┴───────────────┐
                    ▼                              ▼
          3a. escalation event              3b. recovers on its own
          (rapid response / ICU)            (no event: a "false alarm")
          monitor stops: left the ward
                    │
                    ▼
          4. outcome event recorded ──► label "escalated within 6 h?" built later
```

Everything happens on **true physiology**. The data platform only ever sees **measurements** and **outcome events**.

## 3. The hidden rule

Every hour, each patient gets a small probability of **starting** to deteriorate. It rises with:
- **abnormal true vitals** (fast heart or breathing, low oxygen, fever or low temperature, low blood pressure)
- **age**, **number of chronic conditions**, **COPD**, **heart failure**
- a hidden personal **frailty** (random per patient, never observed, like the many things a real hospital doesn't measure)

The weights live only in `generator/truth.py`:

In [ ]:
pd.Series(truth._HIDDEN_WEIGHTS, name="weight").to_frame()

Let's compare a few example patients. The probability is **per hour**, so we also convert it to a chance per day:

In [ ]:
examples = pd.DataFrame(
    {
        "who": ["65, normal vitals", "85, normal vitals", "75 COPD, SpO2 85", "75, fever + low BP"],
        "age": [65, 85, 75, 75],
        "n_conditions": [0, 0, 1, 1],
        "copd": [False, False, True, False],
        "heart_failure": False,
    }
)
hourly = {
    "heart_rate": np.array([[75], [75], [95], [112]], dtype=float),
    "resp_rate": np.array([[16], [16], [24], [24]], dtype=float),
    "spo2": np.array([[97], [97], [85], [95]], dtype=float),
    "temp_c": np.array([[36.8], [36.8], [37.0], [38.9]]),
    "sbp": np.array([[125], [125], [125], [92]], dtype=float),
}
p_hour = truth.hourly_onset_probability(examples, hourly, np.zeros(4), truth.TruthConfig())[:, 0]
examples.assign(
    per_hour=p_hour.round(5),
    per_day=(1 - (1 - p_hour) ** 24).round(3),
)[["who", "per_hour", "per_day"]]

**Interpretation:** a healthy-looking 65-year-old has a tiny chance per day. The hypoxic COPD patient and the feverish, low-blood-pressure patient are many times more likely to start deteriorating.

Notice the rule uses the **true, undisturbed** vitals. In the simulation it's evaluated first, then trajectories are added on top.

## 4. Simulate 3 days of the ward

`truth.simulate_ward` runs everything in order: undisturbed physiology → hidden rule → trajectories → disturbed physiology → measurements.

In [ ]:
%%time
ward = truth.simulate_ward(inpatients, START, hours=72, seed=42)
traj = ward.truth.trajectories
n_events = len(ward.truth.outcomes)
print(f"{len(ward.readings):,} readings | {len(traj)} trajectories | {n_events} escalations")
traj["outcome"].value_counts()

Two tables come out, and they play very different roles:

| Table | Contains | Used by |
|---|---|---|
| `trajectories` | Hidden internals: kind, severity, start time, recovered or not | **Debugging and the case study only.** Restricted audit table, never training. |
| `outcomes` | What the hospital records: patient, encounter, event type, event time | **The data platform**, which builds the labels |

In [ ]:
traj.drop(columns=["patient_id", "encounter_id"]).head(8)

In [ ]:
ward.truth.outcomes.head()

In [ ]:
patient_days = len(ward.readings) / 288
rate = 100 * len(ward.truth.outcomes) / patient_days
print(f"Escalations per 100 patient-days: {rate:.1f}  (target 2-5)")

## 5. Anatomy of one deterioration

Let's follow one **respiratory** deterioration that ended in escalation: true physiology (line), what the monitor sent (dots), trajectory start (orange), escalation (red).

In [ ]:
def show_trajectory(row, hours_before=4, hours_after=1):
    i = inpatients.index[inpatients["patient_id"] == row["patient_id"]][0]
    device = inpatients.loc[i, "device_id"]
    end = (
        row["event_ts"]
        if pd.notna(row["event_ts"])
        else row["start_ts"] + pd.Timedelta(hours=row["planned_hours"] + 4)
    )
    lo, hi = (
        row["start_ts"] - pd.Timedelta(hours=hours_before),
        end + pd.Timedelta(hours=hours_after),
    )
    mask = (ward.timestamps >= lo) & (ward.timestamps <= hi)
    r = ward.readings[(ward.readings["device_id"] == device) & ward.readings["ts"].between(lo, hi)]

    shown = {
        "heart_rate": "Heart rate",
        "resp_rate": "Breathing",
        "spo2": "SpO2",
        "temp_c": "Temp",
        "sbp": "Systolic BP",
    }
    fig, axes = plt.subplots(len(shown), 1, figsize=(11, 8), sharex=True)
    for ax, (v, label) in zip(axes, shown.items(), strict=True):
        ax.plot(ward.timestamps[mask], ward.physiology[v][i, mask], lw=1.5)
        ax.plot(r["ts"], r[v], ".", ms=3, alpha=0.6)
        ax.axvline(row["start_ts"], color="orange", ls="--")
        if pd.notna(row["event_ts"]):
            ax.axvline(row["event_ts"], color="red")
        ax.set_ylabel(label, fontsize=8)
    fig.suptitle(f"{row['kind']} trajectory, severity {row['severity']}, outcome: {row['outcome']}")
    plt.tight_layout()


example = traj.query("outcome == 'escalated' and kind == 'respiratory' and not sudden").iloc[0]
show_trajectory(example)

**Interpretation:** after the orange line, breathing climbs, SpO2 falls and heart rate rises, **slowly at first, then faster** (the progress curve accelerates). At the red line staff escalate and the patient leaves the ward, so **the dots stop**.

The window between orange and red is where a good early-warning model earns its keep.

## 6. Three ways to deteriorate

Each trajectory follows one of three **decompensation profiles**, the change at full severity (`truth.DECOMPENSATION`). COPD patients lean respiratory, heart patients lean cardiac.

In [ ]:
pd.DataFrame(truth.DECOMPENSATION).T.fillna(0)

Sepsis raises **temperature**, and through the fever coupling from notebook 02 that also pushes heart rate and breathing up. Cardiac events drop **blood pressure** sharply.

In [ ]:
sepsis = traj.query("outcome == 'escalated' and kind == 'sepsis' and not sudden")
if len(sepsis):
    show_trajectory(sepsis.iloc[0])

## 7. Recoveries and sudden events

Real medicine is messy, so the generator is too:
- **recoveries (~25%)**: a patient starts deteriorating, peaks part-way, then recovers. No escalation. A model that flags them raises a *false alarm*, and that's unavoidable.
- **sudden events (~10%)**: under 2 hours of warning. No model can catch all of them early.

In [ ]:
recovered = traj.query("outcome == 'recovered'")
if len(recovered):
    show_trajectory(recovered.iloc[0], hours_after=0)

In [ ]:
traj.groupby(["outcome", "sudden"]).size().rename("trajectories").to_frame()

## 8. Building the 6-hour label

The model predicts once per patient per hour: *"will this patient escalate in the next 6 hours?"*
The **label** for hour *t* is 1 if an outcome event happens in **(t, t + 6 h]**, else 0.

Here are the hourly labels for our example patient around their event:

In [ ]:
HORIZON = pd.Timedelta(hours=6)
hours = pd.date_range(START, periods=72, freq="h")

pid = example["patient_id"]
event_ts = ward.truth.outcomes.set_index("patient_id").loc[pid, "event_ts"]
around = hours[(hours >= event_ts - pd.Timedelta(hours=9)) & (hours <= event_ts)]
pd.DataFrame(
    {
        "prediction_hour": around,
        "hours_until_event": ((event_ts - around) / pd.Timedelta(hours=1)).round(1),
        "label": ((event_ts > around) & (event_ts <= around + HORIZON)).astype(int),
        "label_known_at": around + HORIZON,
    }
)

**Interpretation:** the label switches to 1 exactly 6 hours before the event. Look at **`label_known_at`**: at prediction time nobody knows the answer. It only becomes known 6 hours later.
That's the **label delay**. The monitoring system must wait for it before judging the model, and training may only use hours whose 6-hour window has closed.

Over the whole ward, how rare are positive labels?

In [ ]:
events = ward.truth.outcomes.set_index("patient_id")["event_ts"]
grid = pd.MultiIndex.from_product(
    [inpatients["patient_id"], hours], names=["patient_id", "hour"]
).to_frame(index=False)
ev = grid["patient_id"].map(events)
grid = grid[ev.isna() | (grid["hour"] < ev)]  # after escalation the patient has left the ward
ev = ev[grid.index]
grid["label"] = ((ev > grid["hour"]) & (ev <= grid["hour"] + HORIZON)).astype(int)
closed = grid["hour"] + HORIZON <= hours[-1] + pd.Timedelta(hours=1)  # window fully observed
print(f"patient-hours: {len(grid):,} | with a closed 6h window: {closed.sum():,}")
print(
    f"positive labels: {grid.loc[closed, 'label'].sum():,} ({grid.loc[closed, 'label'].mean():.2%})"
)

About **1% of patient-hours are positive**. That's rare, which is why the model needs class weighting and why **AUPRC** (not accuracy) is the main score: a model that always says "no" would be 99% accurate and completely useless.

The last 6 hours of the simulation have **open windows** (we can't know their labels yet), so they're left out. This is the same rule the platform applies.

## 9. Concept drift preview: a new escalation protocol

On day 8 (Plan.md §6.4) the hospital adopts a protocol that **escalates earlier**: at 70% of the way down a trajectory instead of 100%.
The patients and their vitals don't change, but **outcomes for the same vitals do**. That's concept drift.

In [ ]:
results = {}
for name, threshold in {"old protocol (1.0)": 1.0, "new protocol (0.7)": 0.7}.items():
    w = truth.simulate_ward(
        inpatients,
        START,
        hours=72,
        seed=42,
        truth_config=truth.TruthConfig(escalation_threshold=threshold),
    )
    t = w.truth.trajectories
    esc = t[t["outcome"] == "escalated"]
    at_event = []
    for _, row in esc.iterrows():
        i = inpatients.index[inpatients["patient_id"] == row["patient_id"]][0]
        s = w.timestamps.get_loc(row["event_ts"])
        at_event.append({v: w.physiology[v][i, s] for v in ["spo2", "resp_rate", "sbp"]})
    at_event = pd.DataFrame(at_event)
    results[name] = {
        "escalations": len(esc),
        "recoveries": int((t["outcome"] == "recovered").sum()),
        "hours start→event (median)": (
            (esc["event_ts"] - esc["start_ts"]) / pd.Timedelta(hours=1)
        ).median(),
        "SpO2 at event (mean)": at_event["spo2"].mean(),
        "breathing at event (mean)": at_event["resp_rate"].mean(),
    }
pd.DataFrame(results).round(1)

**Interpretation:** under the new protocol there are **more escalations, fewer recoveries**, events happen **sooner**, and at **milder vitals**.
A model trained under the old protocol will now **under-predict** risk for patients who look only moderately unwell. Input-drift monitors won't notice (the vitals look the same). Only **accuracy against the delayed labels** reveals it. That's episode 2 on day 8.

## 10. Exercise: how much warning do we get?

For the **escalated** trajectories in `traj` (the 3-day run from section 4):
1. compute the warning time in hours (`event_ts - start_ts`)
2. what share had **under 2 hours** of warning?
3. what share had **over 6 hours**? (For these, the start of deterioration happens *before* the label even turns positive.)

In [ ]:
# Try it yourself
warning_hours = None

share_under_2h = None
share_over_6h = None

In [ ]:
# Answer (run after trying)
esc = traj[traj["outcome"] == "escalated"]
warning_hours = (esc["event_ts"] - esc["start_ts"]) / pd.Timedelta(hours=1)
share_under_2h = (warning_hours < 2).mean()
share_over_6h = (warning_hours > 6).mean()
print(f"median warning {warning_hours.median():.1f} h")
print(f"under 2 h: {share_under_2h:.0%} | over 6 h: {share_over_6h:.0%}")
warning_hours.plot.hist(bins=12, title="Hours from start of deterioration to escalation");

## 11. Pitfalls and next steps

**Pitfall: training on the audit table.** `trajectories` knows exactly when deterioration started, so using it would leak the answer. The platform only gets **readings** and **outcomes**.

**Pitfall: labelling open windows.** An hour whose 6-hour window hasn't finished has an *unknown* label, not 0. Treating it as 0 teaches the model that recent patients are safe.

**Pitfall: forgetting the monitor stops.** After escalation a patient has no more readings. Hourly features must handle patients leaving (and, soon, new admissions).

**What's still missing:** escalated patients leave, but nobody new is admitted and nobody is discharged, so the ward slowly empties over long runs. The **hospital-activity** step (admissions, discharges, transfers written to the Azure SQL tables) comes before we generate the 14-day baseline.

**Optional extension:** run `truth.simulate_ward(..., truth_config=truth.TruthConfig(recovery_prob=0.5))` and see how the escalation count and false-alarm potential change.

**Next notebook:** `04_drift_scenarios_preview`. All four drift scenarios side by side, and PSI computed by hand.